# 05 · The methods: tree, ladder, bridge, chance reference, fair comparison

**Produces:** the numbers behind your claim, each with its denominator and period, and the chart for slide 4.

**Needs:** notebook 03 run.

Five methods from Weeks 1 and 2, written vectorised so they run on the whole pack in seconds. Use the one your brief needs; read the others so you can follow the other groups on presentation day.

## CONFIG

In [ ]:
import sys, pathlib, time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
bk["half"] = np.where(bk["slot_at"].between(*CONFIG["period_last"]), "H1_2025", np.where(bk["slot_at"].between(*CONFIG["period_this"]), "H1_2026", "other"))
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

### Helpers

Whole-day period filters (`in_period`, `label_half`) and the growth rate `pct`. Assistant use is logged with `logs.ai()` in each cell that used it.

In [ ]:
def in_period(ts, period):
    """Whole days, half-open: on or after the first day, before the day after the last day."""
    return (ts >= pd.Timestamp(period[0])) & (ts < pd.Timestamp(period[1]) + pd.Timedelta(days=1))

def label_half(ts):
    """'H25' or 'H26' for the two half-years, None for anything else."""
    return pd.Series(np.where(in_period(ts, CONFIG["period_last"]), "H25", np.where(in_period(ts, CONFIG["period_this"]), "H26", None)), index=ts.index)

def pct(new, old):
    return round((new / old - 1) * 100, 2)

## Step 1 · Test volume for both halves, checked against the 5 percent

Rebuild tests for H1 2025 and H1 2026 from the clean bookings with whole-day periods (`label_half`), compute the growth rate, and compare it with the 5 percent. Where the figures differ, the cells below find out why before anything else is built on them, and the last cell logs the result.

Done when we hold two totals, one rate, and a note on any difference.

In [ ]:
logs.ai(asked="Step 1 (05): rebuild test volume for both halves and check it against the 5 percent. Gave Claude Code the step and asked for the code",
       did="Claude Code wrote the cell (whole-day halves with label_half); I ran it: 433,611 and 455,383 tests, +5.02%",
       who="Rohit", tool="Claude Code", n=4)
CLAIM_PCT = 5.0   # the growth in test volume the brief asks us to check
tv = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", usecols=["booking_id", "system", "city", "slot_at", "n_tests", "status"], parse_dates=["slot_at"], dtype={"booking_id": "string", "system": "string"})
tv["half"] = label_half(tv["slot_at"])
tv = tv[tv["half"].notna()]
tv_tests = tv.groupby("half")["n_tests"].sum()
tv_growth = pct(tv_tests["H26"], tv_tests["H25"])
checks.check("test volume rebuilt for both halves (whole days)", list(tv_tests.index) == ["H25", "H26"], f"H25 {tv_tests['H25']:,}, H26 {tv_tests['H26']:,}")
print(f"H1 2025  {tv_tests['H25']:>9,} tests")
print(f"H1 2026  {tv_tests['H26']:>9,} tests")
print(f"growth   {tv_growth:+.2f}%   claimed {CLAIM_PCT:+.2f}%   difference {tv_growth - CLAIM_PCT:+.2f} pts")

### What moves the rate: one change at a time

Growth before notebook 03's identity rule, without the two cities that switched booking system, and on collected bookings only, each against the 5 percent.

In [ ]:
# Same whole-day halves, one change at a time: does cleaning, the system switch or the definition of a test move the rate?
logs.ai(asked="Step 1 (05): does cleaning, the system switch or the test definition move the rate? Asked Claude Code for the code",
       did="Claude Code wrote the one-change-at-a-time ladder; I ran it and read the rates (+5.84% before the identity rule, +4.20% collected only)",
       who="Rohit", tool="Claude Code", n=6)
pre = pd.read_csv(CONFIG["clean"] / "bookings.csv.gz", usecols=["slot_at", "n_tests"], parse_dates=["slot_at"])   # before notebook 03's identity rule
pre["half"] = label_half(pre["slot_at"])
switched = sorted(tv.loc[tv["system"].eq("nova"), "city"].unique())   # cities that moved to nova during H1 2026

def per_half(df, label):
    s = df.dropna(subset=["half"]).groupby("half")["n_tests"].sum()
    return pd.Series({"H25": int(s["H25"]), "H26": int(s["H26"]), "growth_pct": pct(s["H26"], s["H25"])}, name=label)

ladder = pd.DataFrame([
    per_half(pre, "before the identity rule (superseded nova records kept)"),
    per_half(tv, "clean, one booking per visit  <- headline"),
    per_half(tv[~tv["city"].isin(switched)], f"clean, without {' and '.join(switched)} (system switch)"),
    per_half(tv[tv["status"].ne("cancelled_or_missing")], "clean, collected bookings only"),
])
ladder["vs_claim_pts"] = (ladder["growth_pct"] - CLAIM_PCT).round(2)
ladder.astype({"H25": int, "H26": int})

### The note on the difference, logged

Prints the note built from the numbers above and writes it once to the challenges log.

In [ ]:
logs.ai(asked="Step 1 (05): write the note on the difference and log it. Asked Claude Code for the code",
       did="Claude Code wrote the note built from the computed numbers and the challenges-log entry; I ran it once",
       who="Rohit", tool="Claude Code", n=7)
pre_growth = ladder["growth_pct"].iloc[0]
NOTE = (f"Rebuilt test volume: H1 2025 {tv_tests['H25']:,}, H1 2026 {tv_tests['H26']:,}, growth {tv_growth:+.2f}% against the claimed {CLAIM_PCT:.0f}% "
        f"({tv_growth - CLAIM_PCT:+.2f} pts). Before notebook 03's identity rule the rate is {pre_growth:+.2f}%.")
print(NOTE)

PHASE = "Methods: test volume vs the 5 percent"
already = PHASE in (logs.DOCS / "challenges-log.md").read_text(encoding="utf-8")
if not already:
    logs.challenge(PHASE,
                   broke=f"Rebuilt growth is {tv_growth:+.2f}% against the claimed {CLAIM_PCT:.0f}%.",
                   tried=f"Rebuilt tests per half from the clean bookings with whole days and before the identity rule ({pre_growth:+.2f}%).",
                   changed=f"Headline uses whole days (label_half): {tv_tests['H25']:,} and {tv_tests['H26']:,} tests, {tv_growth:+.2f}%.",
                   next_time="Check the headline before and after each cleaning rule before building on it.",
                   who="Rohit")
print("Logged to docs/challenges-log.md." if not already else "Already in the challenges log; not appended again.")

## 1 · The tree (brief A)

Tests = people × visits per person × tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

In [ ]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

### The tree by city (cell-2 periods)

Growth of each branch by city on `bk`. Kept for reference; the whole-day versions below are the ones to use.

In [ ]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

### The tree table, whole days

People × visits per person × tests per visit for both halves, with growth, log growth and each branch's share of test growth.

In [ ]:
# The tree table on whole days (label_half), so the totals match Step 1.
logs.ai(asked="Tree (05): drill-down into people x visits per person x tests per visit. Gave Claude Code my target table and asked for the code",
       did="Claude Code wrote the cell reusing tree() on whole days, with log growth and shares; I checked it reproduces my table",
       who="Rohit", tool="Claude Code", n=8)
wd = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", usecols=["booking_id", "person_id", "city", "channel", "slot_at", "n_tests", "rescheduled_from"], parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string"})
wd["half"] = label_half(wd["slot_at"])
wd = wd[wd["half"].notna()]

tw = tree(wd)
branches = ["people", "visits_per_person", "tests_per_visit", "tests"]
tree_table = tw.loc[["H25", "H26"], branches].T.rename(columns={"H25": "H1 2025", "H26": "H1 2026"}).rename_axis(index="branch", columns=None)
tree_table["growth_pct"] = pct(tree_table["H1 2026"], tree_table["H1 2025"])
tree_table["log_growth"] = np.log(tree_table["H1 2026"] / tree_table["H1 2025"])
tree_table["share_pct"] = tree_table["log_growth"] / tree_table.loc["tests", "log_growth"] * 100

checks.check("tests match Step 1 (whole days)", tree_table.loc["tests", "H1 2025"] == tv_tests["H25"] and tree_table.loc["tests", "H1 2026"] == tv_tests["H26"])
checks.check("branch shares add up to 100%", abs(tree_table.loc[branches[:3], "share_pct"].sum() - 100) < 1e-9, f"{tree_table.loc[branches[:3], 'share_pct'].sum():.2f}%")

tree_table.style.format({"H1 2025": lambda v: f"{v:,.0f}" if v > 1000 else f"{v:.4f}", "H1 2026": lambda v: f"{v:,.0f}" if v > 1000 else f"{v:.4f}",
                         "growth_pct": "{:+.2f}%", "log_growth": "{:+.4f}", "share_pct": "{:.0f}%"})

### Two branches against three

Tests per person = visits per person × tests per visit: the growth of each split, and a check that the identity holds.

In [ ]:
# Step 2: two branches against three. Tests per person splits into visits per person x tests per visit.
logs.ai(asked="Tree (05): two branches against three (tests per person). Gave Claude Code my target output and asked for the code",
       did="Claude Code wrote the cell; I checked the output and the tests per person = visits per person x tests per visit check",
       who="Rohit", tool="Claude Code", n=9)
tw["tests_per_person"] = tw["tests"] / tw["people"]
g = pct(tw.loc["H26"], tw.loc["H25"])
print(f"two branches:   people {g['people']:+.2f}%  x  tests per person {g['tests_per_person']:+.2f}%")
print(f"three branches: people {g['people']:+.2f}%  x  visits per person {g['visits_per_person']:+.2f}%  x  tests per visit {g['tests_per_visit']:+.2f}%")
checks.check("Step 2: tests per person = visits per person x tests per visit", np.allclose(tw["tests_per_person"], tw["visits_per_person"] * tw["tests_per_visit"]))

### The tree by city, whole days

Growth of tests, people, tests per person, visits per person and tests per visit in each city.

In [ ]:
# Step 2 by city, on whole days (wd): the three-branch tree with tests per person as the two-branch middle.
logs.ai(asked="Tree (05): drill-down by city. Gave Claude Code my target city table and asked for the code",
       did="Claude Code wrote the cell reusing tree(wd, city); I checked it reproduces my table",
       who="Rohit", tool="Claude Code", n=10)
tc = tree(wd, "city")
tc["tests_per_person"] = tc["tests"] / tc["people"]
cols = ["tests", "people", "tests_per_person", "visits_per_person", "tests_per_visit"]
city_step2 = ((tc.xs("H26", level="half") / tc.xs("H25", level="half") - 1) * 100)[cols].sort_values("tests")
checks.check("Step 2 by city: tests per person = visits per person x tests per visit", np.allclose(tc["tests_per_person"], tc["visits_per_person"] * tc["tests_per_visit"]))
city_step2.style.format("{:+.1f}")

### Actual against plan, by city

The H1 2026 test plan against H1 2025 and H1 2026 actuals: each city's shortfall, its share of the gap, and its share of last year's volume.

In [ ]:
# Step 3: actual against the H1 2026 plan, by city. Plan loaded as in notebook 04 (months melted, Bangalore -> Bengaluru); actuals from tc (whole days).
logs.ai(asked="Plan (05): drill-down into actual against the H1 2026 plan by city. Gave Claude Code my target table and asked for the code",
       did="Claude Code wrote the cell (plan loaded as in notebook 04); I checked it reproduces my table: gap 60,357 tests, Chennai + Jaipur 42.5%",
       who="Rohit", tool="Claude Code", n=11)
plan = pd.read_csv(CONFIG["raw"] / "plan" / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="target_tests")
plan["city"] = plan["city"].replace({"Bangalore": "Bengaluru"})
vs_plan = pd.DataFrame({
    "H1 2025": tc.xs("H25", level="half")["tests"],
    "plan": plan.groupby("city")["target_tests"].sum(),
    "actual H1 2026": tc.xs("H26", level="half")["tests"],
})
checks.check("every city has last year's actual, a plan and this year's actual", bool(vs_plan.notna().all().all()), list(vs_plan.index[vs_plan.isna().any(axis=1)]))
checks.check("city actuals add up to Step 1", vs_plan["H1 2025"].sum() == tv_tests["H25"] and vs_plan["actual H1 2026"].sum() == tv_tests["H26"])

vs_plan["plan growth %"] = pct(vs_plan["plan"], vs_plan["H1 2025"]).round(1)
vs_plan["actual growth %"] = pct(vs_plan["actual H1 2026"], vs_plan["H1 2025"]).round(1)
vs_plan["shortfall"] = vs_plan["plan"] - vs_plan["actual H1 2026"]
vs_plan["share of gap %"] = (vs_plan["shortfall"] / vs_plan["shortfall"].sum() * 100).round(1)
vs_plan["share of 2025 volume %"] = (vs_plan["H1 2025"] / vs_plan["H1 2025"].sum() * 100).round(1)
vs_plan = vs_plan.sort_values("shortfall", ascending=False)
print(vs_plan.to_string())

gap = vs_plan["shortfall"].sum()
print()
print(f"total gap {gap:,} tests = {gap / vs_plan['H1 2025'].sum() * 100:.1f} points of H1 2025 volume")
sw = vs_plan.loc[switched]   # switched: the cities that moved to nova (Step 1)
print(f"{' + '.join(switched)}: {sw['shortfall'].sum() / gap * 100:.1f}% of the gap on {sw['H1 2025'].sum() / vs_plan['H1 2025'].sum() * 100:.1f}% of last year's volume")

### The gap by city and channel, in two orders

The plan has no channel, so it is spread by the H1 2025 mix, city first and channel first, to see whether the order changes who owns the gap.

In [ ]:
# Step 4: the plan gap by city and by channel, split in two orders. The plan has no channel, so each order spreads it by the H1 2025 mix.
logs.ai(asked="Plan (05): split the gap city first and channel first. Gave Claude Code my target table and asked for the code",
       did="Claude Code chose the split that reproduces my table (plan spread by the H1 2025 mix) and wrote the cell; I checked the output",
       who="Rohit", tool="Claude Code", n=12)
cc = wd.groupby(["city", "channel", "half"])["n_tests"].sum().unstack("half")
city_plan = vs_plan["plan"]
# city first: each city's plan, split by that city's H1 2025 channel mix
plan_city_first = city_plan.reindex(cc.index, level="city") * cc["H25"] / cc.groupby(level="city")["H25"].transform("sum")
# channel first: the total plan split by the national H1 2025 channel mix, then each channel's plan across cities by its H1 2025 city mix
channel_plan = city_plan.sum() * cc.groupby(level="channel")["H25"].sum() / cc["H25"].sum()
plan_channel_first = channel_plan.reindex(cc.index, level="channel") * cc["H25"] / cc.groupby(level="channel")["H25"].transform("sum")

def gap_shares(cell_plan):
    """Each city's and each channel's part of the plan gap, as % of the total gap (gap from Step 3)."""
    g = cell_plan - cc["H26"]
    return pd.concat([g.groupby(level="city").sum(), g.groupby(level="channel").sum()]) / gap * 100

orders = pd.DataFrame({"City first": gap_shares(plan_city_first), "Channel first": gap_shares(plan_channel_first)})
orders = orders.loc[list(vs_plan.index) + list(cc.index.unique("channel"))]
checks.check("both orders hand out the whole plan", bool(np.isclose(plan_city_first.sum(), city_plan.sum()) and np.isclose(plan_channel_first.sum(), city_plan.sum())))
checks.check("city shares and channel shares each add up to 100% in both orders", bool(np.allclose(orders.loc[vs_plan.index].sum(), 100) and np.allclose(orders.drop(vs_plan.index).sum(), 100)))
orders.style.format("{:.1f}%")

### Three systems, one question: is the Chennai and Jaipur fall real?

Visits counted by bookings, the collection app and lab results, each against the same quarter last year.

In [ ]:
# Step 5: the same visits counted by three systems, same quarter last year. Only the booking system changed vendor (Chennai and Jaipur, April 2026),
# so if the vendor change bent the count, bookings would part from the collection app and lab results in Q2 26.
logs.ai(asked="Systems (05): is the Chennai and Jaipur fall real? Gave Claude Code my target table of bookings, collection app and lab results and asked for the code",
       did="Claude Code wrote the cell and chose the counting (lab results as distinct bookings by first result) that reproduces my table; I checked the output",
       who="Rohit", tool="Claude Code", n=13)
QUARTERS = {"Q1": ("2025Q1", "2026Q1"), "Q2": ("2025Q2", "2026Q2")}

def quarter_growth(city, ts):
    """Visits per group (switched cities alone, the other four together), % change on the same quarter last year."""
    n = pd.DataFrame({"group": np.where(city.isin(switched), city, "other four"), "q": ts.dt.to_period("Q").astype(str)}).value_counts().unstack("q")
    return pd.DataFrame({f"{k} 26 vs {k} 25": (n[new] / n[old] - 1) * 100 for k, (old, new) in QUARTERS.items()})   # unrounded: pct() rounds, and rounding twice can flip the last digit

# collection app: one row per booking (double scans removed in notebook 03), dated when collected; no-shows have no collected_at and drop out
coll = pd.read_csv(CONFIG["clean"] / "collections_clean.csv.gz", parse_dates=["collected_at"])
coll["city"] = coll["clinic_id"].map(pd.read_csv(CONFIG["raw"] / "reference" / "clinics.csv").set_index("clinic_id")["city"])
# lab results: one row per test run; count each booking once, dated by its first result
res = pd.concat([pd.read_csv(p, usecols=["booking_id", "result_ts"], parse_dates=["result_ts"]).assign(city=p.parent.name.title())
                 for p in sorted((CONFIG["raw"] / "results").glob("*/*.csv.gz"))], ignore_index=True)
first_result = res.groupby("booking_id").agg(city=("city", "first"), result_ts=("result_ts", "min"))
checks.check("every collection maps to a city", bool(coll["city"].notna().all()), int(coll["city"].isna().sum()))

systems = {
    "bookings (vendor changed)": quarter_growth(tv["city"], tv["slot_at"]),
    "collection app": quarter_growth(coll["city"], coll["collected_at"]),
    "lab results": quarter_growth(first_result["city"], first_result["result_ts"]),
}
by_system = (pd.concat(systems, names=["system", "group"]).swaplevel()
               .reindex(pd.MultiIndex.from_product([switched + ["other four"], list(systems)], names=["group", "system"])))
checks.check("every system has a growth rate for every group and quarter", bool(by_system.notna().all().all()))
by_system.style.format("{:+.1f}%")

### Does nova count a visit differently?

Tests per visit, visits per person and the share of visits with no collection record, Q1 to Q2 2026, with and without the reschedule link.

In [ ]:
# Does nova count a visit differently? Chennai and Jaipur moved to nova in Q2 26. If nova split or merged visits, tests per visit, visits per person
# or the share of visits with no collection record would shift there from Q1 to Q2 and not in the other four.
logs.ai(asked="Systems (05): does nova count a visit differently? Gave Claude Code my target table and asked for the code",
       did="Claude Code wrote the cell and found that 1,406 nova reschedules keep their collection under the old booking id; I checked the output",
       who="Rohit", tool="Claude Code", n=14)
q26 = wd[wd["half"].eq("H26")].assign(group=lambda d: np.where(d["city"].isin(switched), d["city"], "other four"),
                                       half=lambda d: d["slot_at"].dt.to_period("Q").astype(str))   # tree() groups on "half"; here it holds the quarter
tq = tree(q26, "group")
checks.check("quarters hold every H1 2026 visit", tq["visits"].sum() == len(q26), f"{tq['visits'].sum():,} visits")

# A visit has a collection record if the app (coll, previous cell) holds its own id, or the id it was rescheduled from:
# nova makes a new booking on reschedule, and the app can keep the collection under the old id, which notebook 03 dropped as superseded.
own = q26["booking_id"].isin(coll["booking_id"])
via_old = q26["rescheduled_from"].isin(coll["booking_id"]) & ~own
tq["no_record_own_id_pct"] = (~own).groupby([q26["group"], q26["half"]]).mean() * 100
tq["no_record_pct"] = (~(own | via_old)).groupby([q26["group"], q26["half"]]).mean() * 100
checks.check("only nova visits carry their collection under the rescheduled-from id", not via_old[q26["group"].eq("other four")].any(),
             f"{int(via_old.sum()):,} visits in {', '.join(switched)}")

fmt = {"tests_per_visit": "{:.3f}", "visits_per_person": "{:.3f}", "no_record_own_id_pct": "{:.1f}%", "no_record_pct": "{:.1f}%"}
nova_check = pd.DataFrame({f"{m} Q1 → Q2": tq[m].unstack("half").apply(lambda r: f"{f.format(r['2026Q1'])} → {f.format(r['2026Q2'])}", axis=1)
                           for m, f in fmt.items()}).loc[switched + ["other four"]]
nova_check

### Verdict: business or recording

In the cities that switched, compares the Q2 fall in bookings with the fall in the collection app.

In [ ]:
# Verdict: in Q2 26 the cities that switched vendor fell in bookings. If the collection app (no vendor change) fell by about as much,
# the fall is in the business; if the two part, the booking system's recording is the suspect.
logs.ai(asked="Systems (05): verdict, business or recording? Gave Claude Code my target output and asked for the code",
       did="Claude Code wrote the cell with a 1-point tolerance; I ran it: same fall, business not recording",
       who="Rohit", tool="Claude Code", n=15)
TOLERANCE_PTS = 1.0   # how far apart the two systems may be and still count as the same fall
q2 = by_system["Q2 26 vs Q2 25"]
for city in switched:
    print(f"{city}: bookings {q2[city, 'bookings (vendor changed)']:+.1f}%, collection app {q2[city, 'collection app']:+.1f}%")
gaps = {city: abs(q2[city, "bookings (vendor changed)"] - q2[city, "collection app"]) for city in switched}
same_fall = all(g <= TOLERANCE_PTS for g in gaps.values())
print("same fall: business, not recording" if same_fall else "bookings and the collection app part: check the recording before calling it business")
checks.check(f"Q2 26: bookings and collection app within {TOLERANCE_PTS} pt in {', '.join(switched)}", same_fall, ", ".join(f"{c} {g:.1f} pts" for c, g in gaps.items()))

### The short branch, by city

People each city would have seen growing at its plan rate, against people actually seen.

In [ ]:
# The short branch, by city: people the city would have seen growing at its plan rate, against people actually seen.
# People from tc (whole days); plan rate = H1 2026 test plan / H1 2025 tests, from vs_plan (Step 3).
logs.ai(asked="Short branch (05): people short by city. Gave Claude Code my target table and asked for the code",
       did="Claude Code wrote the cell (people at the plan rate minus people seen); I checked it against my table (Chennai 3,970 against my 3,971)",
       who="Rohit", tool="Claude Code", n=16)
short = pd.DataFrame({"people H1 2025": tc.xs("H25", level="half")["people"], "people H1 2026": tc.xs("H26", level="half")["people"]})
plan_ratio = vs_plan["plan"] / vs_plan["H1 2025"]
short["plan growth %"] = (plan_ratio - 1) * 100
short["actual growth %"] = (short["people H1 2026"] / short["people H1 2025"] - 1) * 100
short["people short"] = (short["people H1 2025"] * plan_ratio - short["people H1 2026"]).round().astype(int)
short = short.sort_values("people short", ascending=False)
checks.check("every city has people in both halves and a plan", bool(short.notna().all().all()))
print(f"people short in all six cities: {short['people short'].sum():,}")
(short.style.format({"people H1 2025": "{:,}", "people H1 2026": "{:,}", "plan growth %": "{:+.1f}%", "actual growth %": "{:+.1f}%", "people short": "{:,}"})
      .apply(lambda r: ["font-weight: bold" if r.name in switched else ""] * len(r), axis=1))   # the cities that switched vendor

### Who is missing: new or returning

Splits each city's people short by signup date and turns it into extra tests at H1 2026 tests per person.

In [ ]:
# Who is missing: new people (signed up during the half) or returning ones (signed up before it)? Each city's people short splits into the two
# at the same plan rate (plan_ratio, previous cell); extra tests = people short x H1 2026 tests per person in that segment.
logs.ai(asked="Short branch (05): people short split into new and returning. Gave Claude Code my target table and asked for the code",
       did="Claude Code defined new as signed up during the half and wrote the cell; I checked it against my table (small rounding differences)",
       who="Rohit", tool="Claude Code", n=17)
signup = pd.read_csv(CONFIG["clean"] / "patients_clean.csv.gz", usecols=["patient_id", "signup_date"], parse_dates=["signup_date"], dtype={"patient_id": "string"}).set_index("patient_id")["signup_date"]
half_start = wd["half"].map({"H25": pd.Timestamp(CONFIG["period_last"][0]), "H26": pd.Timestamp(CONFIG["period_this"][0])})
seg = wd.assign(segment=np.where(wd["person_id"].map(signup) >= half_start, "new", "returning"))
checks.check("every person has a signup date", bool(wd["person_id"].map(signup).notna().all()))

people = seg.groupby(["city", "segment", "half"])["person_id"].nunique().unstack("half")
city_seg_short = (people["H25"] * plan_ratio.reindex(people.index, level="city") - people["H26"]).round().astype(int)   # whole people, as in the city table
checks.check("segments add up to each city's people short (to rounding)", bool((city_seg_short.groupby(level="city").sum() - short["people short"]).abs().max() <= 1))

ts = tree(seg, "segment").xs("H26", level="half")
segments = pd.DataFrame({"people short": city_seg_short.groupby(level="segment").sum(), "tests per person, H1 2026": ts["tests"] / ts["people"]})
segments["extra tests"] = (segments["people short"] * segments["tests per person, H1 2026"]).round().astype(int)
segments.loc["total"] = [segments["people short"].sum(), np.nan, segments["extra tests"].sum()]
segments.style.format({"people short": "{:,.0f}", "tests per person, H1 2026": "{:.2f}", "extra tests": "{:,.0f}"}, na_rep="")

### The outcome of each visit

Share of visits collected, no-show, and cancelled or missing in each half, with nova reschedules relinked to their collection.

In [ ]:
# Outcome of each visit, per half: collected, no-show, or no collection record (status from notebook 03).
# Nova reschedules keep the collection under the old booking id (see the nova cell), so for those visits use the status the app holds for that id.
logs.ai(asked="Outcomes (05): collected, no-show and cancelled or missing per half. Gave Claude Code my target table and asked for the code",
       did="Claude Code wrote the cell and applied the nova reschedule link to the statuses; I checked it reproduces my table",
       who="Rohit", tool="Claude Code", n=18)
vis = tv.merge(wd[["booking_id", "rescheduled_from"]], on="booking_id", how="left", validate="one_to_one")
checks.join_did_not_fan_out("tv LEFT JOIN wd (rescheduled_from)", tv, vis, "booking_id")
old_status = vis["rescheduled_from"].map(coll.set_index("booking_id")["status"])
relinked = vis["status"].eq("cancelled_or_missing") & old_status.notna()
vis["outcome"] = vis["status"].where(~relinked, old_status)
print(f"visits whose outcome comes from the rescheduled-from booking: {int(relinked.sum()):,}")

labels = {"collected": "Collected", "no_show": "No-show", "cancelled_or_missing": "Cancelled or missing"}
outcomes = (pd.crosstab(vis["half"], vis["outcome"], normalize="index") * 100).rename(index={"H25": "H1 2025", "H26": "H1 2026"}, columns=labels)[list(labels.values())]
checks.check("every visit has one of the three outcomes", bool(vis["outcome"].isin(list(labels)).all()), list(vis.loc[~vis["outcome"].isin(list(labels)), "outcome"].unique()))
checks.check("outcome shares add up to 100% in each half", bool(np.allclose(outcomes.sum(axis=1), 100)))
outcomes.style.format("{:.2f}%")

### Show me, now: the branches multiply back

Rebuilds tests, people, revenue and the gap from their branches for both halves and checks each against the counted total.

In [ ]:
# "Your branches multiply back to the total. Show me, now." Each line rebuilds a total from its branches and checks it against the total counted directly.
logs.ai(asked="Check (05): show that the branches multiply back to the totals for tests, people, revenue and gap. Asked Claude Code for the code",
       did="Claude Code wrote the cell; I ran it and all eight lines pass",
       who="Rohit", tool="Claude Code", n=19)
amount = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", usecols=["booking_id", "amount_rs"], dtype={"booking_id": "string"}).set_index("booking_id")["amount_rs"]
tw["revenue"] = wd["booking_id"].map(amount).groupby(wd["half"]).sum()
tw["rs_per_test"] = tw["revenue"] / tw["tests"]
seg_people = seg.groupby(["half", "segment"])["person_id"].nunique().unstack("segment")

def show(label, parts, op, total):
    """Combine the branches (x or +) and check the result against the directly counted total, to within half a unit."""
    got = np.prod(parts) if op == "x" else np.sum(parts)
    fmt = lambda v: f"{v:,.0f}" if float(v).is_integer() else f"{v:,.2f}" if abs(v) >= 100 else f"{v:,.4f}"   # counts whole, ratios with decimals
    return checks.check(f"{label:<17}", bool(np.isclose(got, total, rtol=0, atol=0.5)), f"{f' {op} '.join(fmt(p) for p in parts)} = {got:,.0f}   (counted {total:,.0f})")

for h in ["H25", "H26"]:
    r = tw.loc[h]
    show(f"tests    {h}", [r["people"], r["visits_per_person"], r["tests_per_visit"]], "x", r["tests"])
    show(f"people   {h}", [seg_people.loc[h, "new"], seg_people.loc[h, "returning"]], "+", r["people"])
    show(f"revenue  {h}", [r["people"], r["visits_per_person"], r["tests_per_visit"], r["rs_per_test"]], "x", r["revenue"])
total_gap = vs_plan["plan"].sum() - vs_plan["actual H1 2026"].sum()
show("gap      cities", list(vs_plan["shortfall"]), "+", total_gap)
show("gap      channels", list((plan_city_first - cc["H26"]).groupby(level="channel").sum()), "+", total_gap)

### Done: branches, owners of the gap, approach log, board sentence

Every branch for both years; the branch and the cities that own most of the gap, with their shares; one approach-log row per join and split; one sentence for the board.

In [ ]:

# 1. Every branch, numbered for both years (tw: whole-day tree, with tests per person and revenue added in the cells above)
rows = ["people", "visits_per_person", "tests_per_visit", "tests_per_person", "tests", "rs_per_test", "revenue"]
branches_both = tw.loc[["H25", "H26"], rows].T.rename(columns={"H25": "H1 2025", "H26": "H1 2026"}).rename_axis(index="branch", columns=None)
branches_both["growth %"] = (branches_both["H1 2026"] / branches_both["H1 2025"] - 1) * 100
num = lambda v: f"{v:,.0f}" if v >= 1000 else f"{v:,.2f}" if v >= 100 else f"{v:.4f}"
display(branches_both.style.format({"H1 2025": num, "H1 2026": num, "growth %": "{:+.2f}%"}))

# 2. Which branch owns the gap? Per city, plan = people at plan rate x H1 2025 tests per person, and actual = people x H1 2026 tests per person, so
#    gap = people short x H1 2026 tests per person  +  people at plan rate x (H1 2025 - H1 2026 tests per person). The two parts add up exactly.
h25, h26 = tc.xs("H25", level="half"), tc.xs("H26", level="half")
people_at_plan = h25["people"] * plan_ratio
gap_branch = pd.DataFrame({"people": (people_at_plan - h26["people"]) * h26["tests_per_person"],
                           "tests per person": people_at_plan * (h25["tests_per_person"] - h26["tests_per_person"])})
checks.check("people branch + tests-per-person branch = each city's shortfall", bool(np.allclose(gap_branch.sum(axis=1), vs_plan["shortfall"].reindex(gap_branch.index))))
branch_share = gap_branch.sum() / gap * 100
print(f"gap {gap:,} tests: " + ", ".join(f"{b} branch {gap_branch[b].sum():+,.0f} tests ({branch_share[b]:.0f}%)" for b in gap_branch))

#    Which cities? The fewest cities, largest first, that together own at least half the gap (vs_plan is sorted by shortfall).
city_share = vs_plan["share of gap %"]
owners = list(vs_plan.index[: int((city_share.cumsum() < 50).sum()) + 1])
owner_share = vs_plan.loc[owners, "shortfall"].sum() / gap * 100
sw_gap = vs_plan.loc[switched, "shortfall"].sum() / gap * 100
sw_vol = vs_plan.loc[switched, "H1 2025"].sum() / vs_plan["H1 2025"].sum() * 100
print("cities owning most of the gap: " + ", ".join(f"{c} {city_share[c]:.1f}%" for c in owners) + f" = {owner_share:.1f}%")
print(f"{' and '.join(switched)} (switched booking system): {sw_gap:.1f}% of the gap on {sw_vol:.1f}% of H1 2025 volume")

# 3. One approach-log row per join and split in this notebook, written once
q2 = by_system["Q2 26 vs Q2 25"]
lab_q2 = ", ".join(f"{g} {q2[g, 'lab results']:+.1f}%" for g in switched + ["other four"])
bk_vs_app = ", ".join(f"{c} {q2[c, 'bookings (vendor changed)']:+.1f}% against {q2[c, 'collection app']:+.1f}%" for c in switched)
cancelled = outcomes["Cancelled or missing"]
city_move = (orders["City first"] - orders["Channel first"]).loc[list(vs_plan.index)].abs().max()


# 4. One sentence for the board
BOARD = (f"Tests grew {tv_growth:.1f}% in H1 2026 ({tv_tests['H25']:,} to {tv_tests['H26']:,}) but fell {gap:,} short of plan; "
         f"the shortfall is people, {short['people short'].sum():,} fewer than plan, worth {gap_branch['people'].sum():,.0f} tests or {branch_share['people']:.0f}% of the gap "
         f"(tests per person moved it by {gap_branch['tests per person'].sum():+,.0f}), "
         f"and {' and '.join(switched)} own {sw_gap:.0f}% of it on {sw_vol:.0f}% of last year's volume, a fall the collection app and lab results show too, so it is the business and not the booking-system switch.")
print(BOARD)